# Consumer View — EDA for Recommendation System

Source: `data/final/consumer_view.csv` (27,902 rows × 54 cols)

Goals:
1. Understand distributions and outliers in each consumer pillar
2. Identify natural segments to anchor recommendations
3. Audit feature availability patterns
4. Assess rating quality and bias
5. Define the feature space for the recommender

In [48]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path('..') 
df = pd.read_csv(BASE / 'data/final/consumer_view.csv')

# Column groups
SPEC_COLS    = ['price', 'mpg_comb', 'hp', 'torque_lbft', 'cargo_cuft']
FEATURE_COLS = [
    'Child Seat Anchors', 'Child Door Locks', 'Traction Control',
    'Stability Control', 'Hill Start Assist', 'Blind-Spot Alert',
    'Collision Warning System', 'Bluetooth Wireless Technology',
    'Hands Free Phone', 'Bluetooth Streaming Audio', 'Satellite Radio',
    'Smartphone Interface', 'Navigation System', 'Voice Recognition System',
    'Internet Access', 'Real-Time Traffic Information', 'Premium Radio',
    'Cruise Control', 'Remote Keyless Entry', 'Remote Engine Start',
    'Power Windows', 'Power Outlet', 'Rear Window Defroster',
    'Steering Wheel Controls', 'Tilt Steering Wheel',
]
RATING_COLS  = ['rating_value', 'rating_performance', 'rating_quality',
                'rating_comfort', 'rating_reliability', 'rating_styling']

print(f'Shape : {df.shape}')
print(f'Years : {df["year"].min()} – {df["year"].max()}')
print(f'Makes : {df["make"].nunique()}')
print(f'Models: {df["model"].nunique()}')

Shape : (27902, 54)
Years : 2000 – 2024
Makes : 64
Models: 896


## 1. Null Audit

In [71]:
null_counts = df.isnull().sum()
null_pct    = (null_counts / len(df) * 100).round(1)
null_df = pd.concat([null_counts.rename('null'), null_pct.rename('null_%')], axis=1)
null_df[null_df['null'] > 0].sort_values('null_%', ascending=False)

,null,null_%
cargo_cuft,4837,17.3
review_id,1814,6.5
Engine,1402,5.0
mpg_city,1183,4.2
mpg_hwy,1183,4.2
mpg_comb,1183,4.2
bodytype,170,0.6
price,170,0.6
price_tier,170,0.6
Drivetrain,1,0.0


## 2. Consumer Segments

Segments anchor the recommendation system — users self-select into one before filtering.

In [50]:
print('=== BODY TYPE ===')
bt = df['bodytype'].value_counts(dropna=False)
print(bt.to_string())
print(f'\nNull: {df["bodytype"].isna().sum():,} ({df["bodytype"].isna().mean():.1%})')

=== BODY TYPE ===
bodytype
SUV            8573
Sedan          7643
Pickup         4674
Coupe          2099
Convertible    1426
Hatchback      1302
Wagon           957
Van             566
Minivan         492
NaN             170

Null: 170 (0.6%)


In [51]:
print('=== FUEL TYPE ===')
print(df['Fuel Type'].value_counts(dropna=False).to_string())
print(f'\nEV share  : {df["is_ev"].mean():.1%}')
print(f'Luxury share: {df["luxury"].mean():.1%}')

=== FUEL TYPE ===
Fuel Type
Gas                   21240
Flex-Fuel              2212
Hybrid                 1823
Diesel                 1327
Electric                904
Plug-in Hybrid          343
Hydrogen Fuel Cell       36
Natural Gas              17

EV share  : 3.2%
Luxury share: 33.0%


In [52]:
# Price tier segmentation
bins   = [0, 15000, 25000, 40000, 60000, 100000, float('inf')]
labels = ['Budget (<15k)', 'Value (15-25k)', 'Mid (25-40k)',
          'Upper-Mid (40-60k)', 'Premium (60-100k)', 'Luxury (100k+)']
df['price_tier'] = pd.cut(df['price'], bins=bins, labels=labels)

tier_counts = df['price_tier'].value_counts().reindex(labels)
tier_pct    = (tier_counts / tier_counts.sum() * 100).round(1)
print('=== PRICE TIERS ===')
pd.concat([tier_counts.rename('count'), tier_pct.rename('%')], axis=1)

=== PRICE TIERS ===


,count,%
price_tier,,
Budget (<15k),13813,49.8
Value (15-25k),7121,25.7
Mid (25-40k),4245,15.3
Upper-Mid (40-60k),1479,5.3
Premium (60-100k),757,2.7
Luxury (100k+),317,1.1


In [53]:
print('=== TOP 20 MAKES (by trim count) ===')
print(df['make'].value_counts().head(20).to_string())

=== TOP 20 MAKES (by trim count) ===
make
Ford             3075
Chevrolet        2027
Toyota           1783
Nissan           1507
BMW              1494
GMC              1464
Audi             1276
Honda            1090
Volkswagen       1049
Mercedes-Benz     968
Jeep              946
Porsche           945
Dodge             830
Subaru            816
Kia               746
Hyundai           725
Mazda             715
Volvo             699
Lexus             646
Land-Rover        598


## 3. Budget Pillar — Price & MPG

In [54]:
print('=== PRICE ===')
print(df['price'].describe().round(0))

# Outlier threshold
p99 = df['price'].quantile(0.99)
print(f'\n99th percentile : ${p99:,.0f}')
print(f'Above 99th      : {(df["price"] > p99).sum():,} rows')

print('\nMedian price by bodytype:')
print(df.groupby('bodytype')['price'].median().sort_values(ascending=False)
        .apply(lambda x: f'${x:,.0f}'))

=== PRICE ===
count     27732.0
mean      20457.0
std       21725.0
min        1125.0
25%        8625.0
50%       15100.0
75%       24950.0
max      687000.0
Name: price, dtype: float64

99th percentile : $106,000
Above 99th      : 277 rows

Median price by bodytype:
bodytype
Coupe          $23,200
Convertible    $20,300
Van            $18,500
SUV            $17,450
Pickup         $14,200
Sedan          $12,800
Hatchback       $9,862
Minivan         $9,800
Wagon           $8,150
Name: price, dtype: object


In [55]:
print('=== MPG_COMB ===')
print(df['mpg_comb'].describe().round(1))

# Extreme values
print(f'\nRows with mpg_comb > 100 (MPGe EVs):')
print(df[df['mpg_comb'] > 100][['make','model','year','Fuel Type','mpg_comb']]
        .sort_values('mpg_comb', ascending=False).head(10).to_string())

print('\nMedian MPG by Fuel Type:')
print(df.groupby('Fuel Type')['mpg_comb'].median().sort_values(ascending=False).round(1))

=== MPG_COMB ===
count    26719.0
mean        24.1
std         12.4
min          9.0
25%         18.0
50%         22.0
75%         26.0
max        142.0
Name: mpg_comb, dtype: float64

Rows with mpg_comb > 100 (MPGe EVs):
          make                                model  year Fuel Type  mpg_comb
22511    Tesla          Model 3 Standard Range Plus  2021  Electric     142.0
20912    Tesla          Model 3 Standard Range Plus  2020  Electric     141.0
25155  Hyundai  Ioniq 6 Long range (18 inch Wheels)  2023  Electric     140.0
26911  Hyundai  Ioniq 6 Long range (18 inch Wheels)  2024  Electric     140.0
27660    Tesla                   Model 3 Long Range  2024  Electric     137.0
15328  Hyundai                       Ioniq Electric  2017  Electric     136.0
16934  Hyundai                       Ioniq Electric  2018  Electric     136.0
18555  Hyundai                       Ioniq Electric  2019  Electric     136.0
26914  Hyundai               Ioniq 6 Standard Range  2024  Electric     135.

## 4. Performance Pillar — HP & Torque

In [56]:
print('=== HORSEPOWER ===')
print(df['hp'].describe().round(0))
print(f'\nhp_missing rate: {df["hp_missing"].mean():.1%}')

print('\nHP outliers (> 800 hp):')
print(df[df['hp'] > 800][['make','model','year','Fuel Type','hp']]
        .sort_values('hp', ascending=False).to_string())

=== HORSEPOWER ===
count    27902.0
mean       279.0
std        106.0
min         27.0
25%        197.0
50%        276.0
75%        329.0
max       1914.0
Name: hp, dtype: float64

hp_missing rate: 11.9%

HP outliers (> 800 hp):
                     make                model  year       Fuel Type      hp
26463       Bugatti Rimac               Nevera  2024        Electric  1914.0
18778          Koenigsegg               Regera  2019          Hybrid  1500.0
20431          Koenigsegg               Regera  2020          Hybrid  1500.0
27260               Lucid                  Air  2024        Electric  1234.0
25527               Lucid                  Air  2023        Electric  1234.0
23834               Lucid                  Air  2022        Electric  1111.0
23835               Lucid                  Air  2022        Electric  1111.0
25524               Lucid                  Air  2023        Electric  1050.0
20618  McLaren Automotive            Speedtail  2020          Hybrid  1036.0
2

In [57]:
print('=== TORQUE ===')
print(df['torque_lbft'].describe().round(0))
print(f'\ntorque_missing rate: {df["torque_missing"].mean():.1%}')

print('\nTorque outliers (> 1500 lb-ft):')
print(df[df['torque_lbft'] > 1500][['make','model','year','Fuel Type','torque_lbft']]
        .sort_values('torque_lbft', ascending=False).to_string())

=== TORQUE ===
count    27902.0
mean       290.0
std        148.0
min          1.0
25%        196.0
50%        266.0
75%        360.0
max      10296.0
Name: torque_lbft, dtype: float64

torque_missing rate: 12.5%

Torque outliers (> 1500 lb-ft):
                make       model  year Fuel Type  torque_lbft
27657          Tesla  Cybertruck  2024  Electric      10296.0
27656          Tesla  Cybertruck  2024  Electric       7435.0
26463  Bugatti Rimac      Nevera  2024  Electric       1740.0


In [58]:
# HP vs price correlation by segment
print('=== HP vs PRICE CORRELATION ===')
corr = df[['price','hp','torque_lbft','mpg_comb']].corr().round(2)
print(corr)

=== HP vs PRICE CORRELATION ===
             price    hp  torque_lbft  mpg_comb
price         1.00  0.55         0.32     -0.04
hp            0.55  1.00         0.70     -0.23
torque_lbft   0.32  0.70         1.00     -0.20
mpg_comb     -0.04 -0.23        -0.20      1.00


## 5. Practicality Pillar — Drivetrain, Transmission & Cargo

In [59]:
print('=== DRIVETRAIN ===')
print(df['Drivetrain'].value_counts(dropna=False).to_string())

print('\n=== TRANSMISSION TYPE ===')
print(df['Transmission Type'].value_counts(dropna=False).to_string())

=== DRIVETRAIN ===
Drivetrain
FWD    7719
AWD    7660
4WD    6003
RWD    4850
2WD    1669
NaN       1

=== TRANSMISSION TYPE ===
Transmission Type
Automatic    25935
Manual        1967


In [60]:
print('=== CARGO SPACE (cu ft) ===')
non_pickup = df[df['bodytype'] != 'Pickup']['cargo_cuft']
print(non_pickup.describe().round(1))

print('\nMedian cargo by bodytype (non-Pickup):')
print(df[df['bodytype'] != 'Pickup']
        .groupby('bodytype')['cargo_cuft']
        .median()
        .sort_values(ascending=False)
        .round(1))

=== CARGO SPACE (cu ft) ===
count    23058.0
mean        45.4
std         50.9
min          3.3
25%         13.8
50%         21.0
75%         64.7
max        463.0
Name: cargo_cuft, dtype: float64

Median cargo by bodytype (non-Pickup):
bodytype
Van            270.4
Minivan        143.8
SUV             66.4
Wagon           55.9
Hatchback       19.0
Sedan           14.9
Coupe           11.5
Convertible      8.8
Name: cargo_cuft, dtype: float64


## 6. Features & Safety

Feature columns use ordinal encoding: `0` = Not Available, `1` = Optional, `2` = Standard.

In [61]:
# Standard rate (== 2) for each feature
std_rate = (df[FEATURE_COLS] == 2).mean().sort_values(ascending=False)
opt_rate = (df[FEATURE_COLS] == 1).mean()
na_rate  = (df[FEATURE_COLS] == 0).mean()

feat_summary = pd.DataFrame({
    'standard_%': (std_rate * 100).round(1),
    'optional_%': (opt_rate * 100).round(1),
    'not_avail_%': (na_rate * 100).round(1),
})
feat_summary

,standard_%,optional_%,not_avail_%
Blind-Spot Alert,24.4,16.7,58.9
Bluetooth Streaming Audio,39.0,1.8,59.2
Bluetooth Wireless Technology,57.8,6.0,36.1
Child Door Locks,67.7,0.0,32.3
Child Seat Anchors,81.9,0.3,17.8
Collision Warning System,21.3,9.5,69.2
Cruise Control,76.1,7.2,16.7
Hands Free Phone,61.5,14.7,23.8
Hill Start Assist,38.5,0.7,60.8
Internet Access,25.5,5.2,69.3


In [62]:
# Standard rate gap: luxury vs non-luxury
lux    = (df[df['luxury'] == 1][FEATURE_COLS] == 2).mean()
nonlux = (df[df['luxury'] == 0][FEATURE_COLS] == 2).mean()
gap = pd.DataFrame({
    'luxury_std%':     (lux    * 100).round(1),
    'non_luxury_std%': (nonlux * 100).round(1),
    'gap':             ((lux - nonlux) * 100).round(1),
}).sort_values('gap', ascending=False)
gap

,luxury_std%,non_luxury_std%,gap
Navigation System,50.7,18.8,31.9
Premium Radio,44.7,16.7,28.0
Bluetooth Wireless Technology,70.5,51.6,18.9
Bluetooth Streaming Audio,47.8,34.6,13.2
Collision Warning System,30.0,17.0,13.0
Traction Control,79.5,66.7,12.8
Real-Time Traffic Information,21.8,9.0,12.8
Hands Free Phone,69.6,57.5,12.1
Satellite Radio,57.7,45.8,11.9
Rear Window Defroster,67.3,56.4,11.0


In [63]:
# Feature co-occurrence: which features tend to come together?
# Use binary (standard or not) for correlation
feat_binary = (df[FEATURE_COLS] == 2).astype(int)
feat_corr = feat_binary.corr().round(2)

# Show top correlated pairs
pairs = (feat_corr.where(np.triu(np.ones(feat_corr.shape), k=1).astype(bool))
                  .stack()
                  .reset_index()
                  .rename(columns={'level_0': 'feature_a', 'level_1': 'feature_b', 0: 'corr'})
                  .sort_values('corr', ascending=False))

print('Top 15 correlated feature pairs:')
# print(pairs.head(15).to_string(index=False))
print(pairs.to_string(index=False))


Top 15 correlated feature pairs:
                    feature_a                     feature_b  corr
Bluetooth Wireless Technology              Hands Free Phone  0.87
             Traction Control             Stability Control  0.77
         Remote Keyless Entry       Steering Wheel Controls  0.73
            Stability Control          Remote Keyless Entry  0.72
               Cruise Control                 Power Windows  0.71
         Remote Keyless Entry         Rear Window Defroster  0.71
             Hands Free Phone       Steering Wheel Controls  0.69
            Stability Control       Steering Wheel Controls  0.69
            Stability Control              Hands Free Phone  0.68
Bluetooth Wireless Technology       Steering Wheel Controls  0.68
Bluetooth Wireless Technology     Bluetooth Streaming Audio  0.68
             Hands Free Phone          Remote Keyless Entry  0.66
            Stability Control Bluetooth Wireless Technology  0.66
        Rear Window Defroster       Steerin

## 7. Review Ratings

In [64]:
print('=== RATING DISTRIBUTIONS ===')
print(df[RATING_COLS].describe().round(2))

print('\n% rated 5.0 (max) per dimension:')
for c in RATING_COLS:
    pct5 = (df[c] == 5).sum() / df[c].notna().sum() * 100
    print(f'  {c:<25} {pct5:.1f}%')

=== RATING DISTRIBUTIONS ===
       rating_value  rating_performance  rating_quality  rating_comfort  \
count      27902.00            27902.00        27902.00        27902.00   
mean           4.26                4.35            4.35            4.33   
std            1.16                1.09            1.17            0.99   
min            1.00                1.00            1.00            1.00   
25%            4.00                4.00            4.00            4.00   
50%            5.00                5.00            5.00            5.00   
75%            5.00                5.00            5.00            5.00   
max            5.00                5.00            5.00            5.00   

       rating_reliability  rating_styling  
count            27902.00        27902.00  
mean                 4.32            4.51  
std                  1.23            0.91  
min                  1.00            1.00  
25%                  4.00            4.00  
50%                  5.00      

In [65]:
print('=== RATING INTER-CORRELATIONS ===')
print(df[RATING_COLS].corr().round(2))

=== RATING INTER-CORRELATIONS ===
                    rating_value  rating_performance  rating_quality  \
rating_value                1.00                0.75            0.80   
rating_performance          0.75                1.00            0.78   
rating_quality              0.80                0.78            1.00   
rating_comfort              0.57                0.66            0.66   
rating_reliability          0.83                0.77            0.87   
rating_styling              0.64                0.69            0.68   

                    rating_comfort  rating_reliability  rating_styling  
rating_value                  0.57                0.83            0.64  
rating_performance            0.66                0.77            0.69  
rating_quality                0.66                0.87            0.68  
rating_comfort                1.00                0.57            0.72  
rating_reliability            0.57                1.00            0.61  
rating_styling         

In [66]:
print('=== RATINGS: LUXURY vs NON-LUXURY (median) ===')
print(df.groupby('luxury')[RATING_COLS].median().round(2).rename({0:'Non-Luxury',1:'Luxury'}))

=== RATINGS: LUXURY vs NON-LUXURY (median) ===
            rating_value  rating_performance  rating_quality  rating_comfort  \
luxury                                                                         
Non-Luxury           5.0                 5.0             5.0             4.0   
Luxury               5.0                 5.0             5.0             5.0   

            rating_reliability  rating_styling  
luxury                                          
Non-Luxury                 5.0             5.0  
Luxury                     5.0             5.0  


## 8. Year Trends

In [67]:
yearly = df.groupby('year').agg(
    trims       = ('car_id',    'count'),
    median_price= ('price',     'median'),
    median_hp   = ('hp',        'median'),
    median_mpg  = ('mpg_comb',  'median'),
    ev_pct      = ('is_ev',     'mean'),
).round(1)
yearly['ev_pct'] = (yearly['ev_pct'] * 100).round(1)
print(yearly[yearly['trims'] >= 10].to_string())

      trims  median_price  median_hp  median_mpg  ev_pct
year                                                    
2000     25       10900.0      263.0        20.0    20.0
2001    386        3262.5      200.0        19.0     0.0
2002    422        3650.0      200.0        20.0     0.0
2003    582        4575.0      215.0        19.0     0.0
2004    676        5150.0      220.0        19.0     0.0
2005    742        5300.0      230.0        19.0     0.0
2006    759        6100.0      245.0        19.0     0.0
2007    791        6325.0      250.0        19.0     0.0
2008    856        7150.0      263.0        19.0     0.0
2009    902        7725.0      265.0        19.0     0.0
2010    902        8275.0      264.0        20.0     0.0
2011   1008        8800.0      270.0        20.0     0.0
2012   1086        9650.0      280.0        21.0     0.0
2013   1252       10600.0      283.0        21.0     0.0
2014   1327       11900.0      285.0        22.0     0.0
2015   1438       12975.0      

## 9. Recommender System Feature Space

Defines which columns to use, how to treat them, and what issues to resolve before modeling.

In [68]:
# Candidate feature columns for the recommender
NUMERIC_FEATURES = ['price', 'mpg_comb', 'hp', 'torque_lbft', 'cargo_cuft']
CATEGORICAL_FEATURES = ['bodytype', 'Fuel Type', 'Drivetrain', 'Transmission Type']
BINARY_FLAGS = ['is_ev', 'luxury']
ORDINAL_FEATURES = FEATURE_COLS   # 0/1/2
RATING_FEATURES = RATING_COLS     # 1.0–5.0, ~12% null

print('Numeric features :', len(NUMERIC_FEATURES))
print('Categorical      :', len(CATEGORICAL_FEATURES))
print('Binary flags     :', len(BINARY_FLAGS))
print('Ordinal (0/1/2)  :', len(ORDINAL_FEATURES))
print('Rating (1–5)     :', len(RATING_FEATURES))
print(f'Total            : {len(NUMERIC_FEATURES)+len(CATEGORICAL_FEATURES)+len(BINARY_FLAGS)+len(ORDINAL_FEATURES)+len(RATING_FEATURES)}')

Numeric features : 5
Categorical      : 4
Binary flags     : 2
Ordinal (0/1/2)  : 25
Rating (1–5)     : 6
Total            : 42


In [69]:
# Issues to handle before modeling
issues = {}

# 1. Price outliers
p99 = df['price'].quantile(0.99)
issues['price outliers (> 99th pct)'] = (df['price'] > p99).sum()

# 2. HP outliers (> 1000)
issues['hp outliers (> 1000)'] = (df['hp'] > 1000).sum()

# 3. Torque outliers (> 2000 lb-ft)
issues['torque outliers (> 2000 lb-ft)'] = (df['torque_lbft'] > 2000).sum()

# 4. MPG outliers (> 100 = MPGe EVs)
issues['mpg_comb > 100 (MPGe)'] = (df['mpg_comb'] > 100).sum()

# 5. Remaining nulls in core numeric features
for c in NUMERIC_FEATURES:
    n = df[c].isna().sum()
    if n > 0:
        issues[f'{c} still null'] = n

# 6. Rating null rows
issues['rows with no ratings'] = df[RATING_COLS[0]].isna().sum()

# 7. bodytype null
issues['bodytype null'] = df['bodytype'].isna().sum()

print('=== PRE-MODELING ISSUES ===')
for k, v in issues.items():
    print(f'  {k:<35} {v:>5,}')

=== PRE-MODELING ISSUES ===
  price outliers (> 99th pct)           277
  hp outliers (> 1000)                   13
  torque outliers (> 2000 lb-ft)          2
  mpg_comb > 100 (MPGe)                 230
  price still null                      170
  mpg_comb still null                 1,183
  cargo_cuft still null               4,837
  rows with no ratings                    0
  bodytype null                         170


In [77]:
# What % of rows have ALL core numeric features filled?
core = ['price', 'mpg_comb', 'hp', 'torque_lbft', 'bodytype', 'Drivetrain', 'Transmission Type']
complete = df[core].notna().all(axis=1)
print(f'Rows with all core features filled: {complete.sum():,} ({complete.mean():.1%})')
print(f'Rows missing at least one core    : {(~complete).sum():,} ({(~complete).mean():.1%})')

Rows with all core features filled: 26,548 (95.1%)
Rows missing at least one core    : 1,354 (4.9%)


count     27902
unique        2
top        True
freq      26548
dtype: object